# Functional Response Reconstruction: ETL, Causal Structure, and Exploratory Analysis

This notebook contains the data extraction, transformation, and exploratory analysis pipeline for reconstructing functional response experiments reported in *Journal of Plankton Research (2026)* on the feeding behavior of *Cordylophora caspia* under varying salinity conditions.

We begin with summary-level data digitized from published figures (mean ± SE, n = 4 per condition), reconstruct the analysis-ready dataset.

---

# 1. ETL: Extract–Transform–Load

The dataset used here was not obtained from raw experimental records. Instead, it was reconstructed from published graphical summaries (Figure 2), where prey consumption was reported as mean ± standard error across four replicates per treatment combination.

The ETL process therefore involves:

1. Digitizing plotted mean points and error bars.
2. Reconstructing per-treatment summary statistics:
   - Mean prey killed
   - Standard error (SE)
   - Sample size (n = 4)
3. Organizing the data into a tidy structure with the following variables:

   - `prey_density` (individuals mL⁻¹)
   - `mean_consumed` (number of prey killed in 2 hours)
   - `se_consumed`
   - `salinity` (10, 20, 30 g L⁻¹)
   - `prey_type` (rotifer, cyclopoid, harpacticoid)

This notebook treats these summary-level observations as data generated from an underlying experimental process. We do not use the fitted curves reported in the original paper.




In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

In [20]:
PROJ_DIR = Path.cwd().parent
DATA_DIR = PROJ_DIR / "data"
EXTR_DAT_DIR = DATA_DIR / "extracted_data"
SENT_DAT_DIR = DATA_DIR / "sent_data"
PROC_DATA_DIR = EXTR_DAT_DIR / "processed"
data_files = list(EXTR_DAT_DIR.glob("figure_2/*.csv"))

In [21]:
def extract_mean_se(group):
    values = np.sort(group["prey_consumed"].values)
    low, mid, high = values
    mean = mid
    se = (high - low) / 2
    return pd.Series({
        "mean_consumed": mean,
        "se_consumed": se,
        "n": 4  # known from paper
    })

In [22]:
def organize_data(file_):
    d = pd.read_csv(file_, header=None, names=["prey_density", "prey_consumed"], usecols=[0, 1])
    d['prey_density'] = d.prey_density.round()
    d['salinity'] = file_.stem.split("_")[1]
    d['prey_type'] = file_.stem.split("_")[2]
    return d

def process_stats(d):
    summary_df = (
        d
        .groupby(["prey_density", "salinity", "prey_type"])
        .apply(extract_mean_se, include_groups=False)
        .reset_index()
        )
    return summary_df

In [23]:
d_final = pd.DataFrame()
for file in data_files:
    try:
        d_interim = process_stats(organize_data(file))
        d_final = pd.concat((d_final, d_interim))
    except Exception as e:
        print(e,file)
d_final.reset_index(drop=True).to_csv("preprocessed_data.csv", index=False)

In [24]:
d_final.reset_index(drop=True, inplace=True)

In [27]:
d_final.to_csv(PROC_DATA_DIR / "d_final.csv")

Comparing to sent data

In [26]:
d_final.head()

,prey_density,salinity,prey_type,mean_consumed,se_consumed,n
0,5.0,20gL,cyclop,2.8125,1.56250,4.0
1,10.0,20gL,cyclop,5.0000,0.46875,4.0
2,15.0,20gL,cyclop,6.2500,0.78125,4.0
3,20.0,20gL,cyclop,9.0625,1.09375,4.0
4,30.0,20gL,cyclop,10.6250,0.62500,4.0


In [16]:
d_sent = pd.read_excel(SENT_DAT_DIR / 'dataset.xlsx')
d_sent.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,Functional Response,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,Bpli,20g,NaN,10g,NaN,30g,NaN,Nito,...,30g,NaN,Apo,NaN,10g,NaN,20g,NaN,30g,NaN
